In [1]:
2+2

4

In [ ]:
import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


# ==========================================================
# Data
# ==========================================================

data = load_iris()

X = data.data
y = data.target

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Xtrain = torch.tensor(Xtrain, dtype=torch.float32)
Xtest = torch.tensor(Xtest, dtype=torch.float32)

ytrain = torch.tensor(ytrain, dtype=torch.long)
ytest = torch.tensor(ytest, dtype=torch.long)


# ==========================================================
# Model
# ==========================================================

class IrisNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 10)
        self.fc2 = nn.Linear(10, 5)
        self.fc3 = nn.Linear(5, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x


model = IrisNet()


# ==========================================================
# Loss / Optimizer
# ==========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


# ==========================================================
# TensorBoard
# ==========================================================

writer = SummaryWriter("runs/iris_demo")


# ----------------------------------------------------------
# Log model graph (once)
# ----------------------------------------------------------
writer.add_graph(model, Xtrain)


# ----------------------------------------------------------
# Log hyperparameters (text)
# ----------------------------------------------------------
writer.add_text(
    "Hyperparameters",
    f"""
    Optimizer : Adam
    Learning Rate : 0.01
    Epochs : 10000
    Loss : CrossEntropy
    Hidden Layers : 10,5
    """
)


# ==========================================================
# Training
# ==========================================================

epochs = 1000
for epoch in range(epochs):
    # ------------------------
    # Training mode
    # ------------------------

    model.train()

    outputs = model(Xtrain)

    loss = criterion(outputs, ytrain)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()


    # ------------------------
    # Metrics
    # ------------------------

    with torch.no_grad():

        model.eval()

        train_pred = model(Xtrain)
        test_pred = model(Xtest)

        train_loss = criterion(train_pred, ytrain)
        test_loss = criterion(test_pred, ytest)

        train_acc = (
            (train_pred.argmax(1) == ytrain)
            .float()
            .mean()
            .item()
        )

        test_acc = (
            (test_pred.argmax(1) == ytest)
            .float()
            .mean()
            .item()
        )


    # ======================================================
    # Scalars
    # ======================================================

    writer.add_scalar(
        "Loss/Train",
        train_loss.item(),
        epoch
    )

    writer.add_scalar(
        "Loss/Test",
        test_loss.item(),
        epoch
    )

    writer.add_scalar(
        "Accuracy/Train",
        train_acc,
        epoch
    )

    writer.add_scalar(
        "Accuracy/Test",
        test_acc,
        epoch
    )

    writer.add_scalar(
        "Learning Rate",
        optimizer.param_groups[0]["lr"],
        epoch
    )


    # ======================================================
    # Parameter Histograms
    # ======================================================

    if epoch % 50 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(
                f"Weights/{name}",
                param,
                epoch
            )


        for name, param in model.named_parameters():

            if param.grad is not None:

                writer.add_histogram(
                    f"Gradients/{name}",
                    param.grad,
                    epoch
                )


    # ======================================================
    # Weight Norms
    # ======================================================

    for name, param in model.named_parameters():

        writer.add_scalar(
            f"Norms/{name}",
            param.norm().item(),
            epoch
        )


    # ======================================================
    # Gradient Norms
    # ======================================================

    for name, param in model.named_parameters():

        if param.grad is not None:

            writer.add_scalar(
                f"Gradient Norms/{name}",
                param.grad.norm().item(),
                epoch
            )


    # ======================================================
    # Parameter Statistics
    # ======================================================

    for name, param in model.named_parameters():

        writer.add_scalar(
            f"Mean/{name}",
            param.mean().item(),
            epoch
        )

        writer.add_scalar(
            f"Std/{name}",
            param.std().item(),
            epoch
        )

        writer.add_scalar(
            f"Max/{name}",
            param.max().item(),
            epoch
        )

        writer.add_scalar(
            f"Min/{name}",
            param.min().item(),
            epoch
        )


    if epoch % 50 == 0:

        writer.add_histogram(
        "Outputs/Train",
        train_pred,
        epoch)

        print(
            f"Epoch {epoch:5d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Test Loss {test_loss:.4f} | "
            f"Train Acc {train_acc:.3f} | "
            f"Test Acc {test_acc:.3f}"
        )


# ==========================================================
# Final metrics
# ==========================================================

writer.add_hparams(
    {
        "optimizer": "Adam",
        "lr": 0.01,
        "epochs": epochs,
        "hidden1": 10,
        "hidden2": 5,
    },
    {
        "Final Train Accuracy": train_acc,
        "Final Test Accuracy": test_acc,
        "Final Train Loss": train_loss.item(),
        "Final Test Loss": test_loss.item(),
    },
)

writer.close()

print(f"\nFinal Test Accuracy = {test_acc * 100:.2f}%")

Epoch     0 | Train Loss 1.3271 | Test Loss 1.3344 | Train Acc 0.333 | Test Acc 0.333
Epoch    50 | Train Loss 0.6027 | Test Loss 0.6028 | Train Acc 0.958 | Test Acc 0.967
Epoch   100 | Train Loss 0.2063 | Test Loss 0.1886 | Train Acc 0.975 | Test Acc 0.967
Epoch   150 | Train Loss 0.1176 | Test Loss 0.1126 | Train Acc 0.983 | Test Acc 0.967
Epoch   200 | Train Loss 0.0903 | Test Loss 0.0843 | Train Acc 0.975 | Test Acc 0.967
Epoch   250 | Train Loss 0.0771 | Test Loss 0.0667 | Train Acc 0.975 | Test Acc 1.000
Epoch   300 | Train Loss 0.0694 | Test Loss 0.0548 | Train Acc 0.975 | Test Acc 1.000
Epoch   350 | Train Loss 0.0642 | Test Loss 0.0464 | Train Acc 0.983 | Test Acc 1.000
Epoch   400 | Train Loss 0.0605 | Test Loss 0.0402 | Train Acc 0.983 | Test Acc 1.000
Epoch   450 | Train Loss 0.0578 | Test Loss 0.0354 | Train Acc 0.983 | Test Acc 1.000
Epoch   500 | Train Loss 0.0558 | Test Loss 0.0318 | Train Acc 0.983 | Test Acc 1.000
Epoch   550 | Train Loss 0.0543 | Test Loss 0.0284 | T

In [8]:
def walk(fn, indent=0):

    if fn is None:
        return

    print(" " * indent, type(fn).__name__)

    for next_fn, _ in fn.next_functions:
        walk(next_fn, indent + 4)

walk(loss.grad_fn)

 NllLossBackward0
     LogSoftmaxBackward0
         AddmmBackward0
             AccumulateGrad
             ReluBackward0
                 AddmmBackward0
                     AccumulateGrad
                     ReluBackward0
                         AddmmBackward0
                             AccumulateGrad
                             TBackward0
                                 AccumulateGrad
                     TBackward0
                         AccumulateGrad
             TBackward0
                 AccumulateGrad
